In [1]:
import torch
import torch.nn as nn
from torchvision.datasets import MNIST
import torchvision.transforms as T

from torch.utils.data import DataLoader
from IPython.display import clear_output
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.optim import Adam
import numpy as np
from PIL import Image
import math
from torch.utils.data import Subset
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [2]:
mnist_transforms = T.Compose(
    [
        T.ToTensor(),
    ]
)

train_dataset = MNIST('mnist', train=True, transform=mnist_transforms, download=True)
valid_dataset = MNIST('mnist', train=False, transform=mnist_transforms, download=True)


np.random.seed(42)
indices_1000 = np.random.choice(len(train_dataset), 1000, replace=False)
train_sub = Subset(train_dataset, indices_1000)

valid_sub = Subset(valid_dataset, range(10000))


train_loader = DataLoader(train_sub, batch_size=64, shuffle=True, pin_memory=True)
valid_loader = DataLoader(valid_sub, batch_size=64, shuffle=False, pin_memory=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 42.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.03MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.6MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]


In [3]:
def train(model):
    model.train()

    train_loss = 0

    for x, _ in tqdm(train_loader, desc='Train'):
        x = x.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = loss_fn(output, x)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_loader)

    return train_loss

In [4]:
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()

    total_loss = 0

    for x, _ in tqdm(loader, desc='Evaluation'):
        x = x.to(device)

        output = model(x)

        loss = loss_fn(output, x)

        total_loss += loss.item()

    total_loss /= len(loader)

    return total_loss

In [5]:
class Block(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, upsample: bool = False):
        super().__init__()
        self.upsample = upsample

        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=1,
            bias=False
        )
        self.norm = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU()

    def forward(self, x):
        if self.upsample:
            x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False, recompute_scale_factor=False)

        return self.act(self.norm(self.conv(x)))


class AutoEncoder(nn.Module):
    def __init__(self, in_channels=1, base_size=32, num_blocks=3, latent_dim=128):
        super().__init__()

        self.base_size = base_size
        self.latent_dim = latent_dim

        encoder_blocks = []
        for i in range(num_blocks):
            encoder_blocks.append(
                Block(
                    in_channels=base_size if i else in_channels,
                    out_channels=base_size,
                    kernel_size=3,
                    stride=2
                )
            )

        self.encoder_conv = nn.Sequential(*encoder_blocks)

        self.flatten = nn.Flatten()
        self.fc_enc = nn.Linear(base_size * 4 * 4, latent_dim)

        self.fc_dec = nn.Linear(latent_dim, base_size * 4 * 4)

        decoder_blocks = []
        for _ in range(num_blocks):
            decoder_blocks.append(
                Block(
                    in_channels=base_size,
                    out_channels=base_size,
                    kernel_size=3,
                    upsample=True
                )
            )

        decoder_blocks.append(nn.Conv2d(base_size, in_channels, kernel_size=3, padding=1))
        decoder_blocks.append(nn.Sigmoid())

        self.decoder = nn.Sequential(*decoder_blocks)

    def forward(self, x):
        x = self.encoder_conv(x)        # -> (bs, 32, 4, 4)
        x = self.flatten(x)
        z = self.fc_enc(x)

        x = self.fc_dec(z)
        x = x.view(-1, self.base_size, 4, 4)
        x = self.decoder(x)             # -> (bs, 1, 32, 32)

        x = x[:, :, 2:30, 2:30]

        return x

    @torch.inference_mode()
    def encode(self, x):
        x = self.encoder_conv(x)
        x = self.flatten(x)
        z = self.fc_enc(x)
        return z


In [6]:
@torch.no_grad()
def get_embeddings_and_labels(model, loader):
    model.eval()
    all_embeddings = []
    all_labels = []

    for x, y in tqdm(loader, desc='Extracting'):
        x = x.to(device)
        z = model.encode(x)

        all_embeddings.append(z.cpu().numpy())
        all_labels.append(y.numpy())

    return np.concatenate(all_embeddings, axis=0), np.concatenate(all_labels, axis=0)


In [7]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(device)
print(torch.cuda.get_device_name())

loss_fn = nn.MSELoss()

cuda:0
Tesla T4


In [8]:
model = AutoEncoder(in_channels=1, latent_dim=128).to(device)

optimizer = Adam(model.parameters(), lr=1e-3)

In [9]:
epochs = 15

for epoch in range(epochs):
    train_loss = train(model)
    valid_loss = evaluate(model, valid_loader)

    print(f"Epoch {epoch+1}: train={train_loss:.4f} valid={valid_loss:.4f}")


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 110.68it/s]


Epoch 1: train=0.1121 valid=0.1415


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 124.67it/s]


Epoch 2: train=0.0555 valid=0.0606


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 116.85it/s]


...

Epoch 13: train=0.0152 valid=0.0166


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 123.43it/s]


Epoch 14: train=0.0147 valid=0.0166


Evaluation: 100%|██████████| 157/157 [00:01<00:00, 123.73it/s]

Epoch 15: train=0.0148 valid=0.0148


In [10]:
train_embeddings, train_labels = get_embeddings_and_labels(model, train_loader)
print(train_embeddings.shape)   # (1000, 128)

test_embeddings, test_labels = get_embeddings_and_labels(model, valid_loader)
print(test_embeddings.shape)    # (10000, 128)


Extracting: 100%|██████████| 16/16 [00:00<00:00, 147.99it/s]


(1000, 128)


Extracting: 100%|██████████| 157/157 [00:01<00:00, 148.33it/s]

(10000, 128)


In [11]:
clf = RandomForestClassifier(random_state=0)
clf.fit(train_embeddings, train_labels)

test_predictions = clf.predict(test_embeddings)
accuracy = accuracy_score(test_labels, test_predictions)

print("Accuracy:", accuracy)



Accuracy: 0.9194
